# SMS Spam Detection using LSTM

## Importing the libraries

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from wordcloud import WordCloud, STOPWORDS

from sklearn.model_selection import train_test_split

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Dense, LSTM

import warnings
warnings.filterwarnings('ignore')

## Dataset

### Loading the dataset

In [ ]:
messages = pd.read_csv("SMSSpamCollection.csv", sep='\t', names=["label", "message"])
messages.head()

In [ ]:
messages.describe()

In [ ]:
duplicatedRow = messages[messages.duplicated()]
duplicatedRow.head()

In [ ]:
messages.groupby('label').describe().T

There are 4,825 ham compared to 747 spam messages. This indicates imbalanced data which we will fix later.

In [ ]:
ham_msg = messages[messages.label == 'ham']
spam_msg = messages[messages.label == 'spam']

ham_msg_text = " ".join(ham_msg.message.to_numpy().tolist())
spam_msg_text = " ".join(spam_msg.message.to_numpy().tolist())

### Plotting the distribution of Spam and Ham messages

In [ ]:
plt.figure(figsize=(8, 6))
sns.countplot(x=messages.label)
plt.title('Distribution of Ham and Spam messages')
plt.xlabel('Message types')
plt.show()

print(f"Spam percentage: {(len(spam_msg) / len(ham_msg)) * 100:.2f}%")

There are more frequent ham messages (85%) than spam (15%). The dataset is imbalanced, so we will apply undersampling to the majority class.

### Down-sampling the majority class

In [ ]:
ham_msg_df = ham_msg.sample(n=len(spam_msg), random_state=44)
spam_msg_df = spam_msg
print(ham_msg_df.shape, spam_msg_df.shape)

#### Plotting the distribution after resampling

In [ ]:
msg_df = pd.concat([ham_msg_df, spam_msg_df]).reset_index(drop=True)
plt.figure(figsize=(8, 6))
sns.countplot(x=msg_df.label)
plt.title('Distribution of Ham and Spam messages (after downsampling)')
plt.xlabel('Message types')
plt.show()

#### Average length of messages

In [ ]:
msg_df['text_length'] = msg_df['message'].apply(len)

labels = msg_df.groupby('label')[['text_length']].mean()
labels

The average length of a spam message is greater than that of a ham message.

## Pre-processing the data

Steps:
1. Converting text labels to numeric values and performing a train-test split.
2. Tokenization.
3. Sequencing and padding.

In [ ]:
msg_df['msg_type'] = msg_df['label'].map({'ham': 0, 'spam': 1})
msg_label = msg_df['msg_type'].values

train_msg, test_msg, train_labels, test_labels = train_test_split(
    msg_df['message'], msg_label, test_size=0.2, random_state=434
)

### Tokenization

In [ ]:
max_len = 50
trunc_type = "post"
padding_type = "post"
oov_tok = "<OOV>"
vocab_size = 500

tokenizer = Tokenizer(num_words=vocab_size, char_level=False, oov_token=oov_tok)
tokenizer.fit_on_texts(train_msg)

word_index = tokenizer.word_index

tot_tokens = len(word_index)
print('No. of unique tokens ==> ', tot_tokens)

### Sequencing and Padding

In [ ]:
training_sequences = tokenizer.texts_to_sequences(train_msg)
training_padded = pad_sequences(training_sequences, maxlen=max_len,
                                padding=padding_type, truncating=trunc_type)

testing_sequences = tokenizer.texts_to_sequences(test_msg)
testing_padded = pad_sequences(testing_sequences, maxlen=max_len,
                               padding=padding_type, truncating=trunc_type)

training_padded = np.array(training_padded)
testing_padded = np.array(testing_padded)
train_labels = np.array(train_labels)
test_labels = np.array(test_labels)

print('Shape of training tensor: ', training_padded.shape)
print('Shape of testing tensor: ', testing_padded.shape)

In [ ]:
print("Before padding ==> ", len(training_sequences[0]))
print("After padding  ==> ", len(training_padded[0]))

## LSTM Model for Spam Detection

### Model Architecture

In [ ]:
vocab_size = 500
embedding_dim = 16
n_lstm = 20
drop_lstm = 0.2

model = Sequential([
    Embedding(vocab_size, embedding_dim, input_length=max_len),
    LSTM(n_lstm, dropout=drop_lstm, return_sequences=True),
    LSTM(n_lstm, dropout=drop_lstm, return_sequences=False),
    Dense(1, activation='sigmoid')
])

model.summary()

### Compiling the Model

In [ ]:
model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

### Training the Model

In [ ]:
num_epochs = 30
early_stop = EarlyStopping(monitor='val_loss', patience=2)

history = model.fit(
    training_padded, train_labels,
    epochs=num_epochs,
    validation_data=(testing_padded, test_labels),
    callbacks=[early_stop],
    verbose=2
)

### Evaluating the Model

In [ ]:
loss, accuracy = model.evaluate(testing_padded, test_labels)
print(f"Test Loss: {loss:.4f}")
print(f"Test Accuracy: {accuracy:.4f}")

## Predict on a Custom Message

Enter any SMS message below to check whether the model classifies it as **spam** or **ham**.

In [ ]:
def predict_message(message):
    """
    Takes a raw SMS message string and returns 'SPAM' or 'HAM'
    along with the model's confidence score.
    """
    sequence = tokenizer.texts_to_sequences([message])
    padded = pad_sequences(sequence, maxlen=max_len, padding=padding_type, truncating=trunc_type)
    padded = np.array(padded)

    score = model.predict(padded, verbose=0)[0][0]

    label = 'SPAM' if score >= 0.5 else 'HAM'
    print(f"Message  : {message}")
    print(f"Result   : {label}")
    print(f"Confidence: {score:.4f} ({'spam probability' if score >= 0.5 else 'ham probability'})")
    return label, score


user_message = input("Enter a message to classify: ")
predict_message(user_message)